# 05 — Student-test evaluation and final model export

Use this notebook only after the baseline comparison and Bayesian optimisation have been run. It evaluates the tuned model on the permitted **student-testing** partition and exports the fitted classifier for integration. The evaluator-testing partition is deliberately not referenced anywhere in this notebook.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             log_loss, confusion_matrix, classification_report)

TRAIN_CSV = Path("data/processed/train_candidates.csv")
STUDENT_CSV = Path("data/processed/student_test_candidates.csv")
MANIFEST = Path("configs/person_b_feature_manifest.json")
OPT = Path("experiments/results/person_b/bayesian_optimization_best.json")
OUT = Path("experiments/results/person_b"); OUT.mkdir(parents=True, exist_ok=True)
ARTIFACTS = Path("artifacts/person_b"); ARTIFACTS.mkdir(parents=True, exist_ok=True)
for p in [TRAIN_CSV, STUDENT_CSV, MANIFEST, OPT]:
    if not p.exists(): raise FileNotFoundError(f"Required file missing: {p}")
train = pd.read_csv(TRAIN_CSV); student = pd.read_csv(STUDENT_CSV)
manifest = json.loads(MANIFEST.read_text()); opt = json.loads(OPT.read_text())
features = manifest["feature_columns"]
X_train, y_train = train[features], train["is_match"].astype(int)
X_test, y_test = student[features], student["is_match"].astype(int)
if set(y_train.unique()) != {0, 1}: raise ValueError("Training target must contain both classes")
if set(y_test.unique()) - {0, 1}: raise ValueError("Student-test target must be binary")
family, params = opt["baseline_family"], opt["best_params"]
if family == "random_forest":
    estimator = RandomForestClassifier(**params, class_weight="balanced_subsample", n_jobs=-1, random_state=42)
elif family == "hist_gradient_boosting":
    estimator = HistGradientBoostingClassifier(**params, random_state=42)
else:
    raise ValueError(f"Unsupported tuned family in export notebook: {family}")
model = make_pipeline(SimpleImputer(strategy="median"), estimator)
start=time.perf_counter(); model.fit(X_train, y_train); fit_seconds=time.perf_counter()-start
start=time.perf_counter(); probabilities=model.predict_proba(X_test)[:,1]; inference_seconds=time.perf_counter()-start
pred=(probabilities >= 0.5).astype(int)
print("Student-test classification report")
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_test, pred, labels=[0,1]))

In [ ]:
# Candidate-level metrics
metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
    "log_loss": log_loss(y_test, probabilities, labels=[0,1]),
    "fit_seconds": fit_seconds,
    "student_test_inference_seconds": inference_seconds,
}

# Observation-level Top-1 accuracy and candidate miss rate.
scored = student[["trip_id", "observation_idx", "candidate_arc_id", "is_match"]].copy()
scored["match_probability"] = probabilities
idx = scored.groupby(["trip_id", "observation_idx"])["match_probability"].idxmax()
top1 = scored.loc[idx]
obs_positive = scored.groupby(["trip_id", "observation_idx"])["is_match"].max()
metrics["observation_top1_accuracy"] = float(top1["is_match"].mean())
metrics["candidate_miss_rate"] = float((obs_positive == 0).mean())
metrics["student_test_observations"] = int(len(obs_positive))
metrics["student_test_trips"] = int(student.trip_id.nunique())
print(json.dumps(metrics, indent=2))
(OUT / "student_test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
scored.to_csv(OUT / "student_test_candidate_probabilities.csv", index=False)

# Refit on all permitted development trips after model selection, then export.
# This is the production model for integration; student-test metrics above are from the model fit on train only.
final_model = make_pipeline(SimpleImputer(strategy="median"), estimator.__class__(**estimator.get_params()))
final_model.fit(pd.concat([X_train, X_test], ignore_index=True), pd.concat([y_train, y_test], ignore_index=True))
joblib.dump({"model": final_model, "feature_columns": features, "target_column": "is_match", "group_column": "trip_id"}, ARTIFACTS / "candidate_classifier.joblib")
metadata = {"model_family": family, "hyperparameters": params, "features": features,
            "trained_on_partitions": ["train", "student_test"],
            "selection_metric": "GroupKFold macro-F1 on training trips",
            "student_test_metrics_from_train_only_fit": metrics,
            "evaluator_test_used": False}
(ARTIFACTS / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Exported final development model to", ARTIFACTS.resolve())

## Evaluation caveats

- Candidate-level metrics and observation-level Top-1 metrics answer different questions; report both.
- Candidate miss rate measures observations for which the candidate generator did not include any positive candidate. A classifier cannot recover a missing candidate.
- Student-test results are for development-stage comparison, not the final evaluator score.
- The exported integration model is refit on train + student-test after model selection. Preserve the train-only student-test metrics separately as shown.
- Do not load, inspect or evaluate evaluator-testing labels in this workflow.